FACEBOOK GRAPH CONSTRUCTION

POST-POST HOMOPHILY GRAPH

In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations

# =====================================================
# LOAD FACEBOOK GRAPH DATASET
# =====================================================

df = pd.read_csv(
    "../data_preprocess/processed_data/graph_data/facebook_graph.csv"
)

print("=" * 60)
print("FACEBOOK GRAPH DATA")
print("=" * 60)

print(df.shape)

FACEBOOK GRAPH DATA
(7527, 32)


In [2]:
G = nx.Graph()

# =====================================================
# ADD POST NODES
# =====================================================

for idx, row in df.iterrows():

    G.add_node(

        row["post_id"],

        # node attributes
        topic=row["topic"],
        language=row["language"],
        media_type=row["media_type"],
        location=row["location"],
        followers=row["followers"],
        popularity=row["popularity"]
    )

print("\nNodes added:", G.number_of_nodes())


Nodes added: 7527


In [ ]:
# =====================================================
# CREATE EDGES
# FACEBOOK HOMOPHILY GRAPH
# =====================================================

from itertools import combinations

post_pairs = combinations(
    df.index,
    2
)

edge_count = 0

for i, j in post_pairs:

    row_i = df.iloc[i]
    row_j = df.iloc[j]

    score = 0

    # =================================================
    # STRONG FACEBOOK SIGNALS
    # =================================================

    # -------------------------------------------------
    # SAME TOPIC
    # very important in Facebook baseline
    # -------------------------------------------------

    if row_i["topic"] == row_j["topic"]:
        score += 2

    # -------------------------------------------------
    # SAME LANGUAGE
    # -------------------------------------------------

    if row_i["language"] == row_j["language"]:
        score += 1

    # -------------------------------------------------
    # SAME MEDIA TYPE
    # -------------------------------------------------

    if row_i["media_type"] == row_j["media_type"]:
        score += 1

    # -------------------------------------------------
    # SAME USER
    # strongest relation
    # -------------------------------------------------

    if row_i["user_id"] == row_j["user_id"]:
        score += 3

    # =================================================
    # OPTIONAL:
    # FOLLOWER SIMILARITY
    # =================================================
    # Similar audience scale
    # helps avoid graph fragmentation
    # =================================================

    follower_ratio = min(
        row_i["followers"],
        row_j["followers"]
    ) / max(
        row_i["followers"],
        row_j["followers"]
    )

    # similar follower scale
    if follower_ratio >= 0.8:
        score += 1

    # =================================================
    # CREATE EDGE
    # =================================================
    # Avoid weak noisy connections
    # =================================================

    if score >= 3:

        G.add_edge(

            row_i["post_id"],
            row_j["post_id"],

            weight=score
        )

        edge_count += 1

print("\nEdges created:", edge_count)